# Adversarial Defenses

Adversarial attacks are easy to run; defenses are hard to make stick. This notebook covers the main categories of defense: adversarial training (the most reliable), input preprocessing defenses (fast but fragile), and certified randomized smoothing (the most principled). We keep training short and use CIFAR-10 with a small CNN so everything runs in a few minutes on a laptop.

In [ ]:
# pip install torch torchvision matplotlib Pillow  # uncomment if needed

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Setup: CIFAR-10 and a Small CNN

We deliberately avoid ResNet here. A small 3-layer CNN trains fast and makes the adversarial training comparison intuitive. The lessons generalize to larger models.

In [ ]:
# CIFAR-10 data loaders
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std  = (0.2471, 0.2435, 0.2616)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(cifar_mean, cifar_std),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(cifar_mean, cifar_std),
])

train_dataset = torchvision.datasets.CIFAR10(root='/tmp/cifar10', train=True,
                                              download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='/tmp/cifar10', train=False,
                                              download=True, transform=test_transform)

# Use a subset to keep training fast
train_subset = torch.utils.data.Subset(train_dataset, range(10000))
test_subset  = torch.utils.data.Subset(test_dataset,  range(2000))

train_loader = torch.utils.data.DataLoader(train_subset, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = torch.utils.data.DataLoader(test_subset,  batch_size=256, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)},  Test batches: {len(test_loader)}')

In [ ]:
class SmallCNN(nn.Module):
    """
    Three convolutional blocks + two fully connected layers.
    Fast enough to train in a few epochs on CPU.
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 32->16
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 16->8
            nn.Dropout2d(0.2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 8->4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters())

model_test = SmallCNN()
print(f'SmallCNN parameters: {count_params(model_test):,}')

## 2. FGSM as a Building Block

We need FGSM as a helper for both the attack evaluation and the adversarial training loop.

In [ ]:
def fgsm_batch(model, x, y, epsilon, loss_fn=nn.CrossEntropyLoss()):
    """
    FGSM on a batch. Returns adversarial batch without gradient tape.
    model must be in the correct mode before calling; this function does not
    change eval/train mode.
    """
    x_adv = x.clone().detach().requires_grad_(True)
    loss = loss_fn(model(x_adv), y)
    loss.backward()
    x_adv = (x_adv.detach() + epsilon * x_adv.grad.sign()).detach()
    return x_adv


def evaluate(model, loader, epsilon=None, device=device):
    """
    Evaluate clean accuracy (epsilon=None) or adversarial accuracy (epsilon given).
    Returns accuracy as a float in [0, 1].
    """
    model.eval()
    correct = 0
    total   = 0
    loss_fn = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if epsilon is not None:
            x = fgsm_batch(model, x, y, epsilon, loss_fn)
        with torch.no_grad():
            preds = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += len(y)

    return correct / total


# Epsilon for CIFAR-10 (pixels in [-1,1] after normalization; 8/255 in pixel space)
EPSILON = 8.0 / 255.0 / np.mean(cifar_std)
print(f'Attack epsilon (normalized): {EPSILON:.4f}')

## 3. Standard Training (Baseline)

Train for 5 epochs on the CIFAR-10 subset without any adversarial examples. This gives our baseline clean and robust accuracy.

In [ ]:
def train_standard(model, loader, optimizer, scheduler, num_epochs, device=device):
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)

        if scheduler:
            scheduler.step()

        avg_loss = running_loss / len(loader.dataset)
        history.append(avg_loss)
        print(f'  Epoch {epoch}/{num_epochs}  loss={avg_loss:.4f}')

    return history


print('Training standard model (5 epochs)...')
t0 = time.time()

standard_model = SmallCNN().to(device)
std_optimizer  = optim.Adam(standard_model.parameters(), lr=1e-3, weight_decay=1e-4)
std_scheduler  = optim.lr_scheduler.CosineAnnealingLR(std_optimizer, T_max=5)

train_standard(standard_model, train_loader, std_optimizer, std_scheduler, num_epochs=5)

std_clean  = evaluate(standard_model, test_loader, epsilon=None)
std_robust = evaluate(standard_model, test_loader, epsilon=EPSILON)

print(f'\nStandard model  |  Clean: {std_clean:.1%}  |  Robust (FGSM): {std_robust:.1%}')
print(f'Training time: {time.time()-t0:.1f}s')

## 4. Adversarial Training

Adversarial training is the most reliable defense we have. The idea is simple: during training, generate adversarial examples on the fly and train the model to classify them correctly.

**The min-max formulation:**

Standard training minimizes the expected loss over the data distribution:
```
min_theta  E[L(x, y; theta)]
```

Adversarial training instead minimizes the worst-case loss inside the epsilon ball:
```
min_theta  E[ max_{||delta||<=eps} L(x + delta, y; theta) ]
```

The inner `max` is the attack (finding the worst perturbation). The outer `min` is the standard gradient descent on model weights.

In practice, we approximate the inner `max` with FGSM (cheap) or PGD (more reliable but slower). Using FGSM makes training roughly 2x slower than standard. Using PGD-10 makes it about 10x slower.

**The cost:** adversarial training almost always reduces clean accuracy by a few percent. The model is spending capacity on robustness that would otherwise go toward clean performance. This is called the *robustness-accuracy tradeoff* and it is a fundamental property, not a bug in the implementation.

In [ ]:
def train_adversarial(model, loader, optimizer, scheduler, num_epochs, epsilon,
                      device=device):
    """
    Adversarial training using FGSM to generate adversarial examples on each batch.
    The model trains on a 50/50 mix of clean and adversarial examples per batch.
    """
    loss_fn = nn.CrossEntropyLoss()
    history = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # Generate adversarial examples for this batch
            # We put model in eval mode temporarily so batch norm uses running stats
            model.eval()
            x_adv = fgsm_batch(model, x, y, epsilon, loss_fn)
            model.train()

            # Train on adversarial examples only (standard approach)
            optimizer.zero_grad()
            loss = loss_fn(model(x_adv), y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y)

        if scheduler:
            scheduler.step()

        avg_loss = running_loss / len(loader.dataset)
        history.append(avg_loss)
        print(f'  Epoch {epoch}/{num_epochs}  loss={avg_loss:.4f}')

    return history


print('Training adversarially (5 epochs)...')
t0 = time.time()

adv_model     = SmallCNN().to(device)
adv_optimizer = optim.Adam(adv_model.parameters(), lr=1e-3, weight_decay=1e-4)
adv_scheduler = optim.lr_scheduler.CosineAnnealingLR(adv_optimizer, T_max=5)

train_adversarial(adv_model, train_loader, adv_optimizer, adv_scheduler,
                  num_epochs=5, epsilon=EPSILON)

adv_clean  = evaluate(adv_model, test_loader, epsilon=None)
adv_robust = evaluate(adv_model, test_loader, epsilon=EPSILON)

print(f'\nAdversarial model  |  Clean: {adv_clean:.1%}  |  Robust (FGSM): {adv_robust:.1%}')
print(f'Training time: {time.time()-t0:.1f}s')

### PGD Adversarial Training: Per-Epoch Clean and Robust Accuracy

The cell above uses FGSM to generate adversarial examples. Here we run a proper PGD-AT loop for 5 epochs and log clean accuracy and robust accuracy at the end of every epoch so we can see how robustness builds over time.

In [ ]:
def pgd_batch(model, x, y, epsilon, alpha, num_steps, loss_fn=nn.CrossEntropyLoss()):
    """
    PGD Linf on a batch. Used inside the adversarial training loop.
    The model's train/eval mode is managed by the caller.
    """
    x_adv = x.clone().detach() + torch.empty_like(x).uniform_(-epsilon, epsilon)
    for _ in range(num_steps):
        x_adv.requires_grad_(True)
        loss = loss_fn(model(x_adv), y)
        loss.backward()
        x_adv = (x_adv.detach() + alpha * x_adv.grad.sign()).detach()
        x_adv = torch.max(torch.min(x_adv, x + epsilon), x - epsilon)
    return x_adv.detach()


def evaluate_robust(model, loader, epsilon, pgd_steps, device=device):
    """Return (clean_acc, fgsm_acc, pgd_acc) on a loader."""
    loss_fn = nn.CrossEntropyLoss()
    clean_c = adv_fgsm_c = adv_pgd_c = total = 0
    alpha = epsilon * 0.375   # 2.4/255 at eps=8/255 (typical ratio)

    model.eval()
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        with torch.no_grad():
            clean_c += (model(x).argmax(1) == y).sum().item()

        x_fgsm = fgsm_batch(model, x, y, epsilon)
        with torch.no_grad():
            adv_fgsm_c += (model(x_fgsm).argmax(1) == y).sum().item()

        x_pgd = pgd_batch(model, x, y, epsilon, alpha, pgd_steps)
        with torch.no_grad():
            adv_pgd_c += (model(x_pgd).argmax(1) == y).sum().item()

        total += len(y)

    return clean_c / total, adv_fgsm_c / total, adv_pgd_c / total


def train_pgd_at(num_epochs=5, pgd_steps_train=7, device=device):
    """
    PGD adversarial training loop on CIFAR-10.
    Uses PGD-7 to generate adversarial examples each batch.
    Logs clean and robust accuracy after each epoch.
    """
    model = SmallCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    loss_fn   = nn.CrossEntropyLoss()
    alpha_train = EPSILON * 0.375

    history = []
    print(f'{"Epoch":>5}  {"Train Loss":>10}  {"Clean Acc":>10}  {"FGSM Acc":>9}  {"PGD-7 Acc":>9}')
    print('-' * 55)

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # Generate PGD adversarial examples (model in eval for BN stability)
            model.eval()
            x_adv = pgd_batch(model, x, y, EPSILON, alpha_train, pgd_steps_train, loss_fn)
            model.train()

            optimizer.zero_grad()
            loss = loss_fn(model(x_adv), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(y)

        scheduler.step()
        avg_loss = running_loss / len(train_loader.dataset)

        # Evaluate on a small subset to keep it fast
        clean_a, fgsm_a, pgd_a = evaluate_robust(
            model, test_loader, EPSILON, pgd_steps=7
        )
        history.append({'epoch': epoch, 'loss': avg_loss,
                         'clean': clean_a, 'fgsm': fgsm_a, 'pgd': pgd_a})
        print(f'{epoch:>5}  {avg_loss:>10.4f}  {clean_a:>10.1%}  {fgsm_a:>9.1%}  {pgd_a:>9.1%}')

    return model, history


print('Running PGD-AT training for 5 epochs (this takes a few minutes on CPU)...')
t0 = time.time()
pgd_at_model, pgd_at_history = train_pgd_at(num_epochs=5, pgd_steps_train=7)
print(f'\nTotal training time: {time.time() - t0:.1f}s')

# --- Plot clean vs. robust accuracy per epoch ---
epochs   = [h['epoch'] for h in pgd_at_history]
clean_h  = [h['clean'] for h in pgd_at_history]
fgsm_h   = [h['fgsm']  for h in pgd_at_history]
pgd_h    = [h['pgd']   for h in pgd_at_history]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, clean_h, 'o-', label='Clean accuracy',    color='steelblue')
ax.plot(epochs, fgsm_h,  's--', label='FGSM accuracy',   color='darkorange')
ax.plot(epochs, pgd_h,   '^:', label='PGD-7 accuracy',   color='firebrick')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('PGD Adversarial Training: accuracy per epoch\n(CIFAR-10 subset, SmallCNN, eps=8/255)')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/pgd_at_history.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side comparison
labels = ['Standard model', 'Adversarial model']
clean_accs  = [std_clean,  adv_clean]
robust_accs = [std_robust, adv_robust]

x = np.arange(2)
width = 0.3

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, clean_accs,  width, label='Clean accuracy',  color='steelblue')
bars2 = ax.bar(x + width/2, robust_accs, width, label='Robust accuracy (FGSM eps=8/255)', color='firebrick')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Accuracy')
ax.set_title('Clean vs. Robust accuracy: Standard vs. Adversarial training\n(CIFAR-10 subset, 5 epochs, SmallCNN)')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('adv_training_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nRobustness gain:  {adv_robust - std_robust:+.1%}')
print(f'Clean accuracy cost: {adv_clean - std_clean:+.1%}')

## 5. Input Preprocessing Defenses

A simpler approach: before feeding the image to the model, apply a transformation that destroys the adversarial perturbation. This requires no retraining.

Common preprocessing defenses:
- **JPEG compression:** lossy compression discards high-frequency components, which is where many adversarial perturbations live.
- **Gaussian smoothing:** averages neighboring pixels, blurring the perturbation.
- **Bit-depth reduction:** round pixel values to fewer bits, which quantizes away small perturbations.

**The catch:** these are all easily adaptive-attacked. If the attacker knows you're applying JPEG compression, they can generate adversarial examples that survive JPEG compression. Preprocessing defenses are *not* robust against an adaptive adversary. They raise the bar slightly but do not provide guarantees.

Still, they are worth understanding because they are cheap and sometimes used in production as a first line of defense.

In [ ]:
def jpeg_defense(x_normalized, quality=75):
    """
    Apply JPEG compression to a normalized batch tensor.
    Works image-by-image via PIL.
    """
    mean = torch.tensor(cifar_mean).view(3, 1, 1)
    std  = torch.tensor(cifar_std).view(3, 1, 1)
    result = []
    for img in x_normalized.cpu():
        # Denormalize to [0, 1]
        pixel = (img * std + mean).clamp(0, 1)
        # Convert to uint8 PIL image
        pil_img = T.ToPILImage()(pixel)
        # JPEG round-trip
        buf = io.BytesIO()
        pil_img.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        pil_jpeg = Image.open(buf).convert('RGB')
        # Re-normalize
        tensor = T.ToTensor()(pil_jpeg)
        tensor = T.Normalize(cifar_mean, cifar_std)(tensor)
        result.append(tensor)
    return torch.stack(result).to(x_normalized.device)


def gaussian_defense(x_normalized, sigma=1.0):
    """
    Gaussian blur applied in normalized space.
    """
    kernel_size = 3  # must be odd
    return TF.gaussian_blur(x_normalized, kernel_size=[kernel_size, kernel_size],
                            sigma=[sigma, sigma])


def bit_depth_defense(x_normalized, bits=5):
    """
    Reduce pixel values to `bits`-bit precision.
    Operates by denormalizing, quantizing, renormalizing.
    """
    mean = torch.tensor(cifar_mean).view(1, 3, 1, 1).to(x_normalized.device)
    std  = torch.tensor(cifar_std).view(1, 3, 1, 1).to(x_normalized.device)
    pixel = (x_normalized * std + mean).clamp(0, 1)      # [0,1]
    levels = 2 ** bits
    pixel_q = (pixel * levels).round() / levels           # quantize
    return (pixel_q - mean) / std                         # renormalize


def evaluate_with_defense(model, loader, epsilon, defense_fn, device=device):
    """
    Generate FGSM adversarial examples, apply defense_fn, then evaluate.
    Also returns clean accuracy under the defense (to measure clean cost).
    """
    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    results = {'clean_defended': (0, 0), 'robust_defended': (0, 0)}

    clean_correct = adv_correct = total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Clean + defense
        x_def = defense_fn(x)
        with torch.no_grad():
            clean_correct += (model(x_def).argmax(1) == y).sum().item()

        # Adversarial + defense
        x_adv = fgsm_batch(model, x, y, epsilon, loss_fn)
        x_adv_def = defense_fn(x_adv)
        with torch.no_grad():
            adv_correct += (model(x_adv_def).argmax(1) == y).sum().item()

        total += len(y)

    return clean_correct / total, adv_correct / total


print('Preprocessing defense utilities defined.')

In [ ]:
print('Evaluating preprocessing defenses on the standard model...')
print('(This runs FGSM on each batch, so it takes a moment)')
print()

defenses = {
    'No defense':         lambda x: x,
    'JPEG (quality=75)':  lambda x: jpeg_defense(x, quality=75),
    'JPEG (quality=50)':  lambda x: jpeg_defense(x, quality=50),
    'Gaussian blur':      lambda x: gaussian_defense(x, sigma=1.0),
    'Bit-depth (5 bits)': lambda x: bit_depth_defense(x, bits=5),
}

defense_results = {}
for name, fn in defenses.items():
    clean_d, robust_d = evaluate_with_defense(standard_model, test_loader, EPSILON, fn)
    defense_results[name] = (clean_d, robust_d)
    print(f'{name:<25}  Clean: {clean_d:.1%}  |  Robust: {robust_d:.1%}')

### JPEG Compression Defense

JPEG is a lossy format: it discards high-frequency components that are invisible to the human eye. Many adversarial perturbations live in exactly that frequency band, so JPEG compression at quality 75 often destroys a meaningful fraction of the perturbation.

Below we measure accuracy before and after JPEG compression at quality 75 on FGSM adversarial examples.

In [ ]:
print('JPEG compression defense: measuring accuracy before and after, on FGSM adversarial examples')
print()

# Standard model: no defense (baseline for adversarial examples)
no_def_clean, no_def_adv = evaluate_with_defense(standard_model, test_loader, EPSILON, lambda x: x)
jpeg_clean_75, jpeg_adv_75 = evaluate_with_defense(
    standard_model, test_loader, EPSILON, lambda x: jpeg_defense(x, quality=75)
)

print(f'{"Defense":<30}  {"Clean acc":>10}  {"FGSM acc":>10}  {"Adv improvement":>16}')
print('-' * 72)
print(f'{"No defense":<30}  {no_def_clean:>10.1%}  {no_def_adv:>10.1%}  {"--":>16}')
print(f'{"JPEG quality=75":<30}  {jpeg_clean_75:>10.1%}  {jpeg_adv_75:>10.1%}  '
      f'{jpeg_adv_75 - no_def_adv:>+15.1%}')

print()
print('Key question: how much of the original FGSM adversarial budget survives JPEG compression?')
print('The improvement tells you how destructive JPEG is to the perturbation.')

### Gaussian Noise Smoothing

Adding independent Gaussian noise to the input before feeding it to the model can disrupt adversarial perturbations at the cost of some clean accuracy. Higher sigma destroys more of the perturbation but also makes clean images harder to classify.

We sweep sigma over [0.1, 0.25, 0.5] to see the accuracy trade-off.

In [ ]:
def gaussian_noise_defense(x, sigma):
    """Add independent Gaussian noise with standard deviation sigma."""
    return x + torch.randn_like(x) * sigma


sigma_values = [0.1, 0.25, 0.5]
gauss_results = []

print(f'{"Sigma":<8}  {"Clean acc":>10}  {"FGSM acc":>10}')
print('-' * 34)

for sigma in sigma_values:
    def_fn = lambda x, s=sigma: gaussian_noise_defense(x, s)
    c_acc, r_acc = evaluate_with_defense(standard_model, test_loader, EPSILON, def_fn)
    gauss_results.append((sigma, c_acc, r_acc))
    print(f'{sigma:<8.2f}  {c_acc:>10.1%}  {r_acc:>10.1%}')

# Plot the trade-off
sigmas   = [r[0] for r in gauss_results]
c_accs   = [r[1] for r in gauss_results]
r_accs   = [r[2] for r in gauss_results]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sigmas, c_accs, 'o-', label='Clean accuracy',         color='steelblue')
ax.plot(sigmas, r_accs, 's--', label='Robust (FGSM) accuracy', color='firebrick')
ax.set_xlabel('Noise sigma')
ax.set_ylabel('Accuracy')
ax.set_title('Gaussian noise defense: accuracy trade-off vs. sigma\n(standard model, CIFAR-10 subset)')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/gauss_noise_tradeoff.png', dpi=100, bbox_inches='tight')
plt.show()

### Input Transformation Defense: Random Crop and Resize

A spatial transformation (randomly crop then resize back to original dimensions) changes the pixel coordinates of the perturbation. Because FGSM perturbations are pixel-aligned, this spatial shift can partially break them.

The defense: apply `RandomCrop` with a small padding, then `Resize` back to 32x32. The perturbation is now spatially shifted and interpolated, which degrades it.

In [ ]:
import torchvision.transforms.functional as TF_func
import random as py_random

def random_crop_resize_defense(x, crop_size=28, final_size=32):
    """
    For each image in the batch:
      1. Randomly crop to (crop_size x crop_size)
      2. Resize back to (final_size x final_size)
    The random offset shifts the adversarial perturbation spatially.
    """
    B, C, H, W = x.shape
    out = []
    for img in x:
        # Random crop offset
        top  = py_random.randint(0, H - crop_size)
        left = py_random.randint(0, W - crop_size)
        cropped = TF_func.crop(img, top, left, crop_size, crop_size)
        resized = TF_func.resize(cropped, [final_size, final_size],
                                  interpolation=TF_func.InterpolationMode.BILINEAR,
                                  antialias=True)
        out.append(resized)
    return torch.stack(out)


# Evaluate random-crop-resize defense
crop_clean, crop_adv = evaluate_with_defense(
    standard_model, test_loader, EPSILON,
    lambda x: random_crop_resize_defense(x, crop_size=28, final_size=32)
)

print(f'Random crop-resize defense (28->32):')
print(f'  Clean accuracy: {crop_clean:.1%}')
print(f'  FGSM accuracy:  {crop_adv:.1%}  (baseline without defense: {no_def_adv:.1%})')
print()
print('Note: The randomness means each call gives slightly different results.')
print('In practice this defense is applied multiple times and the predictions are averaged.')

## RobustBench: Loading a Pretrained Robust Model

RobustBench (Croce et al., 2021) is a standardized benchmark that provides pretrained robust models for CIFAR-10, CIFAR-100, and ImageNet. Loading one takes a single function call.

Install with: `pip install robustbench`

We load `Engstrom2019Robustness`, a ResNet-50 adversarially trained with PGD, and compare it against our locally trained standard model.

In [ ]:
# pip install robustbench  # uncomment to install

try:
    from robustbench.utils import load_model as rb_load_model

    # Download and load the Engstrom2019Robustness model (ResNet-50, CIFAR-10, Linf)
    robust_rb_model = rb_load_model(
        model_name='Engstrom2019Robustness',
        dataset='cifar10',
        threat_model='Linf',
    ).to(device).eval()

    print(f'RobustBench model loaded: Engstrom2019Robustness')
    print(f'Parameters: {sum(p.numel() for p in robust_rb_model.parameters()):,}')

    # Evaluate RobustBench model vs. our local standard model
    # Note: RobustBench models expect inputs in [0,1]; we need to pass unnormalized data.
    # We create a thin wrapper that applies CIFAR-10 normalization inside.
    class CifarNormWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
            mean = torch.tensor(cifar_mean).view(1, 3, 1, 1)
            std  = torch.tensor(cifar_std).view(1, 3, 1, 1)
            self.register_buffer('mean', mean)
            self.register_buffer('std',  std)

        def forward(self, x_01):
            return self.model((x_01 - self.mean) / self.std)

    # The RobustBench model already handles normalization internally.
    # Evaluate directly on our normalized test loader by computing in [0,1] space.
    def eval_rb_model(model, loader, epsilon_01, device=device):
        """Evaluate a RobustBench model. loader returns normalized CIFAR-10 tensors."""
        loss_fn = nn.CrossEntropyLoss()
        clean_c = fgsm_c = pgd_c = total = 0
        alpha_01 = epsilon_01 * 0.375

        # Denorm helper
        mean_t = torch.tensor(cifar_mean, device=device).view(1, 3, 1, 1)
        std_t  = torch.tensor(cifar_std,  device=device).view(1, 3, 1, 1)

        model.eval()
        for x_norm, y in loader:
            x_norm, y = x_norm.to(device), y.to(device)
            x_01 = (x_norm * std_t + mean_t).clamp(0, 1)

            with torch.no_grad():
                clean_c += (model(x_01).argmax(1) == y).sum().item()

            # FGSM in [0,1] space
            x_f = x_01.clone().detach().requires_grad_(True)
            loss = loss_fn(model(x_f), y)
            loss.backward()
            x_fgsm_01 = (x_f.detach() + epsilon_01 * x_f.grad.sign()).clamp(0, 1).detach()
            with torch.no_grad():
                fgsm_c += (model(x_fgsm_01).argmax(1) == y).sum().item()

            # PGD-20 in [0,1] space
            x_p = x_01.clone().detach()
            for _ in range(20):
                x_p.requires_grad_(True)
                loss = loss_fn(model(x_p), y)
                loss.backward()
                x_p = (x_p.detach() + alpha_01 * x_p.grad.sign()).detach()
                x_p = torch.max(torch.min(x_p, x_01 + epsilon_01), x_01 - epsilon_01).clamp(0, 1)
            with torch.no_grad():
                pgd_c += (model(x_p).argmax(1) == y).sum().item()

            total += len(y)

        return clean_c / total, fgsm_c / total, pgd_c / total

    eps_01 = 8.0 / 255.0   # epsilon in [0,1] pixel space
    rb_clean, rb_fgsm, rb_pgd20 = eval_rb_model(robust_rb_model, test_loader, eps_01)

    # Also get numbers for our local standard model using the same evaluation
    std_clean_rb, std_fgsm_rb, std_pgd20_rb = eval_rb_model(
        CifarNormWrapper(standard_model).to(device), test_loader, eps_01
    )

    print()
    print(f'{"Model":<30}  {"Clean":>7}  {"FGSM":>7}  {"PGD-20":>8}')
    print('-' * 60)
    print(f'{"Standard (our SmallCNN)":<30}  {std_clean_rb:>7.1%}  {std_fgsm_rb:>7.1%}  {std_pgd20_rb:>8.1%}')
    print(f'{"Engstrom2019 (RobustBench)":<30}  {rb_clean:>7.1%}  {rb_fgsm:>7.1%}  {rb_pgd20:>8.1%}')

except ImportError:
    print('robustbench not installed. Run:  pip install robustbench')
    print()
    print('Example code that would run:')
    print("""
    from robustbench.utils import load_model

    robust_model = load_model(
        model_name='Engstrom2019Robustness',
        dataset='cifar10',
        threat_model='Linf',
    )
    """)
except Exception as e:
    print(f'RobustBench demo failed: {e}')

In [ ]:
# Plot
names  = list(defense_results.keys())
clean_vals  = [defense_results[n][0] for n in names]
robust_vals = [defense_results[n][1] for n in names]

x = np.arange(len(names))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, clean_vals,  width, label='Clean accuracy',  color='steelblue', alpha=0.85)
ax.bar(x + width/2, robust_vals, width, label='Robust accuracy (FGSM)', color='firebrick', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Preprocessing defenses: clean vs. robust accuracy (standard model)')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('preprocessing_defenses.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. Randomized Smoothing

Preprocessing defenses are heuristics. They work against specific attacks but not against an adaptive adversary who knows the defense.

Randomized smoothing (Cohen et al., 2019) is different: it provides a *certified* guarantee. For a given input, you can compute a certified radius `r` such that the model's prediction is guaranteed to be the same for all perturbations within that radius.

**The idea:**

Define a *smoothed classifier* g as:
```
g(x) = argmax_c  P[f(x + noise) = c]   where noise ~ N(0, sigma^2 * I)
```

In words: add Gaussian noise to the input many times, run the base classifier `f` on each noisy version, and take the majority vote.

Cohen et al. showed that if the majority class wins with probability `p_A >= 0.5`, then the smoothed classifier is certified robust within radius:
```
r = sigma * Phi_inv(p_A)
```
where `Phi_inv` is the inverse CDF of the standard normal.

**The tradeoffs:**
- Higher `sigma` gives larger certified radii, but makes the base classifier work harder (noisy inputs are hard).
- The certified accuracy (what fraction of the test set is correctly classified *and* certified) is always lower than clean accuracy.
- Inference is slow: you need many forward passes per input to estimate `p_A`.

We will not implement the full certification procedure here because it requires hundreds of forward passes per image. Instead, here is what the code structure looks like conceptually:

In [ ]:
from scipy import stats

def smoothed_predict(model, x, sigma, num_samples=100):
    """
    Conceptual implementation of randomized smoothing prediction.
    Returns the predicted class and an estimate of p_A (the winning probability).

    In a real certified system you would use much more samples (e.g., 100,000)
    and apply a Clopper-Pearson confidence interval to get a statistical guarantee.
    """
    model.eval()
    num_classes = 10  # CIFAR-10

    # Draw num_samples noisy versions of x
    x_rep = x.repeat(num_samples, 1, 1, 1)   # [num_samples, C, H, W]
    noise = torch.randn_like(x_rep) * sigma
    x_noisy = x_rep + noise

    with torch.no_grad():
        logits = model(x_noisy.to(device))
        preds  = logits.argmax(dim=1).cpu()  # [num_samples]

    # Count votes
    counts = torch.bincount(preds, minlength=num_classes).float()
    top_class = counts.argmax().item()
    p_A = (counts[top_class] / num_samples).item()

    # Certified radius (only valid if p_A > 0.5)
    if p_A > 0.5:
        radius = sigma * stats.norm.ppf(p_A)
    else:
        radius = 0.0  # abstain

    return top_class, p_A, radius


# Demo on a single CIFAR-10 test image
test_iter = iter(test_loader)
x_batch, y_batch = next(test_iter)
x_single = x_batch[0:1]  # just one image
y_single  = y_batch[0].item()

cifar_classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

for sigma in [0.12, 0.25, 0.50]:
    pred, p_A, radius = smoothed_predict(standard_model, x_single, sigma, num_samples=200)
    true_label  = cifar_classes[y_single]
    pred_label  = cifar_classes[pred]
    print(f'sigma={sigma:.2f}  pred={pred_label:<12}  true={true_label:<12}  '
          f'p_A={p_A:.2f}  certified_radius={radius:.4f}')

## 7. Robustness as a Design Goal

The results so far tell a consistent story:

- Preprocessing defenses give small improvements in robust accuracy at essentially no cost, but they break immediately against an adaptive attack.
- Adversarial training gives genuine robust accuracy, but it costs clean accuracy and training time.
- Certified defenses (randomized smoothing) provide guarantees, but at a significant clean accuracy cost and slow inference.

**Retrofitting vs. building in:** If you train a model for production without thinking about robustness, and then try to add adversarial training later, you will usually need to retrain from scratch. The loss landscape of a trained model is already shaped around clean examples; adversarial training changes the geometry of what the model learns. You cannot bolt robustness on after the fact without retraining.

This is the core engineering lesson: if adversarial robustness matters for your application, it must be a design goal from day one, not an afterthought.

**When does it matter?**
- Safety-critical computer vision: medical imaging, autonomous driving, surveillance
- Anywhere the input comes from an untrusted source and a motivated attacker might manipulate it
- Physical-world deployments where patch attacks are feasible

For most internal-use classifiers where inputs are not adversarially controlled, standard training is fine. Know your threat model before spending resources on robustness.

## 8. Hands-on: Full Training Comparison

Let's put all the numbers together in one clean comparison plot.

In [ ]:
# Compile all results
comparison = {
    'Standard (no defense)':   (std_clean,  std_robust),
    'Adv. training (FGSM)':    (adv_clean,  adv_robust),
    'Std + JPEG q=75':         defense_results.get('JPEG (quality=75)', (std_clean, std_robust)),
    'Std + Gaussian blur':     defense_results.get('Gaussian blur', (std_clean, std_robust)),
    'Std + Bit-depth (5bit)':  defense_results.get('Bit-depth (5 bits)', (std_clean, std_robust)),
}

names        = list(comparison.keys())
clean_vals   = [comparison[n][0] for n in names]
robust_vals  = [comparison[n][1] for n in names]

x = np.arange(len(names))
width = 0.3

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, clean_vals,  width, label='Clean accuracy',  color='steelblue', alpha=0.85)
ax.bar(x + width/2, robust_vals, width, label='Robust accuracy (FGSM eps=8/255)', color='firebrick', alpha=0.85)

for i, (cv, rv) in enumerate(zip(clean_vals, robust_vals)):
    ax.text(i - width/2, cv + 0.005, f'{cv:.1%}', ha='center', va='bottom', fontsize=7)
    ax.text(i + width/2, rv + 0.005, f'{rv:.1%}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Defense comparison: Clean vs. Robust accuracy (CIFAR-10 subset, SmallCNN)')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('full_defense_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nSummary table:')
print(f'{"Defense":<30}  {"Clean":>7}  {"Robust":>7}  {"Gap":>7}')
print('-' * 55)
for name in names:
    cv, rv = comparison[name]
    print(f'{name:<30}  {cv:>7.1%}  {rv:>7.1%}  {cv-rv:>7.1%}')

## Evaluation Table: Clean, FGSM, and PGD-20 Accuracy for Each Defense

We consolidate all defense methods into one table: clean accuracy, FGSM accuracy, and PGD-20 accuracy. PGD-20 is a stronger attack than FGSM and reveals which defenses collapse under iterative pressure.

In [ ]:
def evaluate_all_three(model, loader, epsilon, defense_fn=lambda x: x, pgd_steps=20, device=device):
    """
    Return (clean_acc, fgsm_acc, pgd_acc) for a model with an optional defense.
    Defense is applied AFTER generating the adversarial example.
    """
    loss_fn = nn.CrossEntropyLoss()
    alpha   = epsilon * 0.375
    clean_c = fgsm_c = pgd_c = total = 0

    model.eval()
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Clean + defense
        with torch.no_grad():
            clean_c += (model(defense_fn(x)).argmax(1) == y).sum().item()

        # FGSM adversarial + defense
        x_f = fgsm_batch(model, x, y, epsilon, loss_fn)
        with torch.no_grad():
            fgsm_c += (model(defense_fn(x_f)).argmax(1) == y).sum().item()

        # PGD adversarial + defense
        x_p = x.clone().detach()
        for _ in range(pgd_steps):
            x_p.requires_grad_(True)
            loss = loss_fn(model(x_p), y)
            loss.backward()
            x_p = (x_p.detach() + alpha * x_p.grad.sign()).detach()
            x_p = torch.max(torch.min(x_p, x + epsilon), x - epsilon)
        with torch.no_grad():
            pgd_c += (model(defense_fn(x_p)).argmax(1) == y).sum().item()

        total += len(y)

    return clean_c / total, fgsm_c / total, pgd_c / total


# Collect results for each configuration
table_rows = []

configs = [
    ('Standard model, no defense',        standard_model, lambda x: x),
    ('Standard + JPEG q=75',              standard_model, lambda x: jpeg_defense(x, quality=75)),
    ('Standard + Gaussian noise (s=0.1)', standard_model, lambda x: gaussian_noise_defense(x, 0.1)),
    ('Standard + Crop-Resize (28->32)',   standard_model, lambda x: random_crop_resize_defense(x, 28, 32)),
    ('FGSM Adv. training',               adv_model,       lambda x: x),
    ('PGD-7 Adv. training',              pgd_at_model,    lambda x: x),
]

print(f'{"Defense / Model":<38}  {"Clean":>7}  {"FGSM":>7}  {"PGD-20":>8}')
print('=' * 66)

for name, mdl, def_fn in configs:
    c, f, p = evaluate_all_three(mdl, test_loader, EPSILON, def_fn, pgd_steps=20)
    table_rows.append({'name': name, 'clean': c, 'fgsm': f, 'pgd20': p})
    print(f'{name:<38}  {c:>7.1%}  {f:>7.1%}  {p:>8.1%}')

print()
print('Takeaway: preprocessing defenses improve FGSM accuracy but PGD-20 collapses them.')
print('Only adversarial training gives genuine PGD-20 robustness.')

## Exercise

**Exercise 1: PGD adversarial training**
The training loop above uses FGSM to generate adversarial examples. Replace FGSM with PGD (5 steps, alpha=2/255). Does robust accuracy improve? By how much? What is the training time overhead compared to FGSM-based adversarial training?

**Exercise 2: Mixed training**
Instead of training on adversarial examples only, train on a 50/50 mix of clean and adversarial examples per batch. Modify `train_adversarial` to concatenate `x` and `x_adv` before computing the loss. Compare clean and robust accuracy to the pure adversarial training case. Does the clean accuracy cost decrease?

**Exercise 3: Adaptive preprocessing attack**
Pick the JPEG defense (quality=75). Write an "adaptive" FGSM that generates adversarial examples which are robust to JPEG compression:
1. Apply JPEG to the adversarial example inside the gradient computation.
2. Compute the loss on the JPEG-compressed adversarial example.
3. Take the gradient of that loss with respect to the original input.

How much of the JPEG defense's gain does this adaptive attack recover?

Hint: you will need `jpeg_defense` to be differentiable, or you can approximate it with a Gaussian blur (which is differentiable).

**Exercise 4: Epsilon sweep on adversarially trained model**
Evaluate both models (standard and adversarially trained) at epsilon values of 4/255, 8/255, 16/255, 32/255. Plot robust accuracy vs. epsilon for both models. At what epsilon does the adversarially trained model break down?

## Exercise: Randomized Smoothing Prediction

Implement a randomized smoothing prediction function. The idea: add Gaussian noise to the input N times, run the base classifier on each noisy copy, take the majority vote as the predicted class, and estimate the certified radius from the winning probability.

In [ ]:
from scipy import stats as scipy_stats

def randomized_smoothing_predict(model, x, sigma, num_samples=200, num_classes=10):
    """
    Randomized smoothing prediction (Cohen et al., 2019).

    Samples N noisy versions of x, runs the model on each, takes majority vote.
    If p_A > 0.5, returns a certified radius under Gaussian (L2) perturbation.

    Args:
        model:       PyTorch classifier in eval mode
        x:           single image tensor (1, C, H, W), normalized
        sigma:       noise standard deviation
        num_samples: how many noisy copies to sample
        num_classes: number of output classes

    Returns:
        predicted_class: int, the majority-vote class
        p_A:             float, estimated winning probability
        certified_radius: float, certified L2 radius (0.0 if p_A <= 0.5)
    """
    # YOUR CODE HERE
    #
    # Steps:
    # 1. Create x_rep = x.repeat(num_samples, 1, 1, 1)
    # 2. Add Gaussian noise: noise = torch.randn_like(x_rep) * sigma
    # 3. Forward pass (no_grad): logits = model(x_rep + noise)
    # 4. preds = logits.argmax(dim=1)   -> shape (num_samples,)
    # 5. counts = torch.bincount(preds, minlength=num_classes).float()
    # 6. predicted_class = counts.argmax().item()
    # 7. p_A = counts[predicted_class] / num_samples
    # 8. If p_A > 0.5:  certified_radius = sigma * scipy_stats.norm.ppf(p_A.item())
    #    Else:          certified_radius = 0.0   (abstain)
    raise NotImplementedError


# --- Test your implementation ---
# Once implemented, this block should print certified radii for multiple sigma values.

test_batch_iter = iter(test_loader)
x_test_batch, y_test_batch = next(test_batch_iter)
x_single_test = x_test_batch[0:1].to(device)
y_true_test   = y_test_batch[0].item()

cifar_classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(f'True label: {cifar_classes[y_true_test]}')
print()

for sigma in [0.12, 0.25, 0.50]:
    try:
        pred_class, p_A, radius = randomized_smoothing_predict(
            standard_model, x_single_test, sigma=sigma, num_samples=200
        )
        print(f'sigma={sigma:.2f}  pred={cifar_classes[pred_class]:<12}  '
              f'p_A={p_A:.2f}  certified_radius={radius:.4f}')
    except NotImplementedError:
        print(f'sigma={sigma:.2f}  [not implemented yet]')

print()
print('Note: for a proper certification guarantee, use num_samples >= 10000')
print('and apply a Clopper-Pearson confidence interval on p_A.')